# General GPT-2 Attention Analysis
Paper-aligned streaming analysis of **What Does BERT Look At?**, Sections 3.2–3.3:
https://arxiv.org/abs/1906.04341.

The original `General_Analysis.ipynb` is the reference for the Figure 1, 2 and 4 layouts.
All layer/head indices below are **one-based**. Numerical differences from BERT are
expected: GPT-2 is causal, context length is at most 1024, and a natural paragraph
boundary is two newline tokens rather than a BERT separator.

**Correction of the earlier run:** encoding the separator separately forced ID 628.
The v2 input tokenizes the joined paragraphs normally. The old ID-628 claim is withdrawn.
See `results/gpt2_streaming/protocol.md` for the frozen tests and discovery/validation split.
The initial 158 articles are discovery; the remaining 842 are validation. Two input
conditions are evaluated: boundary-wrapped text and the same text without leading EOT.
Only attention statistics are measured; none establishes functional no-op behavior.


In [ ]:
import csv
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

from gpt2_streaming import (
    GROUPS, FIXED_HEADS, stream_measurements, export_tables, bootstrap_mean)
sns.set_style("darkgrid")  # match the original General_Analysis.ipynb
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "pdf.fonttype": 42, "savefig.dpi": 300})
BLACK, RED, BLUE = "k", "#e74c3c", "#3498db"
PURPLE, GREEN, ORANGE = "#9b59b6", "#159d82", "#f39c12"
INPUT_FILE = "data/wiki_gpt2_natural_v2.jsonl"
RESULTS_DIR = Path("results/gpt2_streaming")
FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(name):
    for extension in ("png", "pdf"):
        plt.gcf().savefig(FIGURE_DIR / (name + "." + extension),
                          bbox_inches="tight", dpi=300)

def points(matrix):
    return np.repeat(np.arange(1, 13), 12), matrix.reshape(-1)

def head_scatter(ax, matrix, color, label, mean=True):
    x, y = points(matrix)
    ax.scatter(x, y, color=color, s=12, alpha=0.65, label=label)
    if mean:
        ax.plot(np.arange(1, 13), matrix.mean(-1), color=color, linewidth=2)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("Layer")
    ax.set_xlim(0.6, 12.4)


## Loading / streaming extraction
Run this notebook from the repository root. If the compact archives already exist,
this cell loads them without another model pass. Otherwise each attention map is
consumed in RAM and released after computing per-article statistics; only a few
illustration heads are retained. The two runs use identical article/paragraph choices.


In [ ]:
runs = stream_measurements(INPUT_FILE, RESULTS_DIR)
main = runs["canonical"]
control = runs["no_leading_eot"]
n_docs = len(main["length"])
validation_n = max(n_docs - 158, 0)
manifest = json.loads((RESULTS_DIR / "stream_manifest.json").read_text())
print("N =", n_docs, "| discovery =", min(n_docs, 158),
      "| validation =", validation_n)
print("Lengths:", int(main["length"].min()), "to", int(main["length"].max()))
print("Ordinary queries:", int(main["query_count"].sum()))
print("Compact archives:", sum(p.stat().st_size for p in RESULTS_DIR.glob("*.npz")),
      "bytes")
tests = export_tables(main, control, RESULTS_DIR)
entropies = main["raw_entropy"].mean(0)  # raw entropy retained for Section 6
normalized_entropies = main["normalized_entropy"].mean(0)


## Experiment 1 compatibility: relative position
Current/previous-token statistics remain available for Ben. The next-token value
is structurally zero for a causal decoder. Document means are averaged equally.


In [ ]:
avg_attns = {
    "self": main["self"].mean(0),
    "right": main["previous"].mean(0),  # legacy plot convention: previous
    "left": np.zeros((12, 12)),        # next is causally masked
    "cls": main["category_all"][..., GROUPS.index("initial")].mean(0),
    "sep": main["category_all"][..., GROUPS.index("boundary")].mean(0),
    "punct": main["category_all"][..., GROUPS.index("punct")].mean(0),
    "sep_sep": main["boundary_from_boundary"].mean(0),
    "rest_sep": main["boundary_from_other"].mean(0),
}


## Experiment 2 — Figure 2: where does attention go?
As in the original notebook, points represent heads and lines represent layer means.
The first panel includes self attention and all query rows. The second uses
boundary versus non-boundary ordinary queries. These are descriptive GPT-2
analogues, **not** claims that natural newlines have BERT's SEP semantics.
The third preserves the relative-position panel. A later diagnostic restricts
queries to positions that can actually see the paragraph boundary.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(6.8, 10), sharex=True)
for key, color, label in [
        ("cls", RED, "initial EOT"),
        ("sep", BLUE, "natural paragraph boundary"),
        ("punct", PURPLE, ". or ,")]:
    head_scatter(axes[0], avg_attns[key], color, label)
axes[0].set_title("Attention to token categories")
axes[0].annotate("L08H03", (8, avg_attns["cls"][7, 2]),
                 xytext=(12, -4), textcoords="offset points", fontsize=8)
axes[0].set_ylabel("Avg. attention")
axes[0].set_ylim(-0.02, 1.02)
axes[0].legend(fontsize=8, loc="upper left")

for key, color, label in [
        ("rest_sep", BLUE, "other → boundary"),
        ("sep_sep", GREEN, "boundary → boundary")]:
    head_scatter(axes[1], avg_attns[key], color, label)
axes[1].set_title("Boundary queries versus other queries (self included)")
axes[1].set_ylabel("Avg. attention")
axes[1].set_ylim(-0.02, 1.02)
axes[1].legend(fontsize=8)

for key, color, label in [
        ("left", RED, "next token"),
        ("right", BLUE, "previous token"),
        ("self", PURPLE, "current token")]:
    head_scatter(axes[2], avg_attns[key], color, label, mean=False)
axes[2].annotate("L05H12", (5, avg_attns["right"][4, 11]),
                 xytext=(10, -4), textcoords="offset points", fontsize=8)
axes[2].set_ylabel("Avg. attention")
axes[2].set_ylim(-0.02, 1.02)
axes[2].legend(fontsize=8)
fig.suptitle(f"GPT-2 Heads — Figure 2 analogue (N={n_docs})", y=1.01)
fig.tight_layout()
save_figure("figure2_token_categories")
plt.show()
plt.close(fig)


## Experiment 3 — Figure 4: focused versus broad attention
This restores the original two-panel raw-entropy layout. The top panel shows
ordinary queries; the bottom shows the trailing EOT query. Unlike BERT's CLS,
GPT-2's initial EOT can see only itself, so its entropy is uninformative. The
trailing EOT is a visible-support comparison, not a supervised CLS equivalent.
Each panel has its own **mean causal-uniform** reference. No future smoothing
is applied. Articles have equal weight.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6.8, 7), sharex=True)
for ax, metric, reference, color, label in [
        (axes[0], "raw_entropy", "uniform_raw", BLUE, "GPT-2 Heads"),
        (axes[1], "trailing_raw_entropy", "trailing_uniform_raw", RED,
         "GPT-2 Heads from trailing EOT")]:
    head_scatter(ax, main[metric].mean(0), color, label)
    ax.axhline(main[reference].mean(), color=BLACK, linestyle="--",
               label="causal-uniform reference")
    ax.set_ylabel("Avg. attention entropy (nats)")
    ax.legend(fontsize=8, loc="upper right")
fig.suptitle(f"GPT-2 Heads — Figure 4 analogue (N={n_docs})")
fig.tight_layout()
save_figure("figure4_raw_entropy")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
head_scatter(axes[0], normalized_entropies, BLUE, "ordinary queries")
head_scatter(axes[0], main["trailing_normalized_entropy"].mean(0), RED,
             "trailing EOT")
axes[0].axhline(1, color=BLACK, linestyle="--")
axes[0].set_ylabel("Causal-normalized entropy")
axes[0].legend(fontsize=8)
head_scatter(axes[1], main["late_max_attention"].mean(0), PURPLE, "q ≥ 32")
axes[1].axhline(0.1, color=BLACK, linestyle="--", label="10% threshold")
axes[1].set_ylabel("Mean largest attention weight")
axes[1].legend(fontsize=8)
fig.suptitle(f"GPT-2 Heads — broadness beyond raw entropy (N={n_docs})")
fig.tight_layout()
save_figure("entropy_and_max_weight")
plt.show()
plt.close(fig)


## Fixed-head validation and distance controls
The eight hypotheses were recorded before rerunning the model. The table uses
only the 842 unexamined articles. Bootstrap intervals are for the mean effect;
the exact sign test concerns the direction across articles. Holm correction
treats all eight tests as one family. Negative outcomes are retained.

Distance matching redistributes each query/head's observed attention *within*
each distance bucket. It thus preserves that query's local-distance preference.
Self links and EOT queries are excluded from target enrichment.


In [ ]:
for row in tests:
    print(f"{row['hypothesis']}: {row['description']}\n"
          f"  effect={row['mean_effect']:.4f} "
          f"95% CI [{row['ci_low']:.4f}, {row['ci_high']:.4f}]; "
          f"positive documents={row['positive_documents']}/{row['nonzero_documents']}; "
          f"Holm p={row['holm_p']:.3g}; supported={row['supported']}")

validation = slice(158, None)
selected = [(2, 6, "period", "L03H07 → periods"),
            (10, 3, "boundary", "L11H04 → boundary"),
            (2, 6, "newline", "L03H07 → newlines")]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (l, h, group, title) in zip(axes, selected):
    c = GROUPS.index(group)
    observed = main["past_mass"][validation, l, h, c]
    matched = main["distance_expected"][validation, l, h, c]
    for x, values, color in [(0, observed, BLUE), (1, matched, GREY if "GREY" in globals() else "#888888")]:
        mean, lo, hi = bootstrap_mean(values)
        ax.bar(x, mean, color=color, alpha=0.85)
        ax.errorbar(x, mean, yerr=[[mean-lo], [hi-mean]], color="k", capsize=4)
    ax.set_xticks([0, 1], ["Observed", "Distance matched"])
    ax.set_ylabel("Mean past attention mass")
    ax.set_title(title)
fig.suptitle(f"GPT-2 Heads — fixed candidates on validation (N={validation_n})")
fig.tight_layout()
save_figure("validation_target_controls")
plt.show()
plt.close(fig)


## What causes the concentrated heads?
Compare initial-key attention with and without a leading EOT. This input
comparison changes both prefix identity and absolute position; it cannot isolate
their causal contributions. The self-attention diagnostic shows whether low
entropy comes from copying the current position. The conditional-entropy panel
removes initial-key mass and renormalizes the remaining distribution *as a
statistical diagnostic*, not as a model intervention.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
l, h = 7, 2
for x, run, label, color in [(0, main, "Leading EOT", RED),
                             (1, control, "First content\n(no leading EOT)", BLUE)]:
    mean, lo, hi = bootstrap_mean(run["initial"][validation, l, h])
    axes[0].bar(x, mean, color=color)
    axes[0].errorbar(x, mean, yerr=[[mean-lo], [hi-mean]], color="k", capsize=4)
axes[0].set_xticks([0, 1], ["Leading EOT", "First content\n(no leading EOT)"])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Attention to position 0")
axes[0].set_title("L08H03: position-zero concentration")

for x, metric, label in [(0, "self", "Self"), (1, "initial", "Initial EOT"),
                          (2, "previous", "Previous")]:
    mean, lo, hi = bootstrap_mean(main[metric][validation, 4, 11])
    axes[1].bar(x, mean, color=[PURPLE, RED, BLUE][x])
    axes[1].errorbar(x, mean, yerr=[[mean-lo], [hi-mean]], color="k", capsize=4)
axes[1].set_xticks([0, 1, 2], ["Self", "Initial EOT", "Previous"])
axes[1].set_ylim(0, 1)
axes[1].set_title("L05H12: why is entropy near zero?")
axes[1].set_ylabel("Mean attention mass")

head_scatter(axes[2], main["late_normalized_entropy"].mean(0), BLUE,
             "original distribution")
head_scatter(axes[2], np.nanmean(main["initial_removed_entropy"], axis=0),
             ORANGE, "conditioned on key ≠ 0")
axes[2].set_title("Entropy after excluding initial-key mass")
axes[2].set_ylabel("Normalized entropy (q ≥ 32)")
axes[2].legend(fontsize=7)
fig.suptitle(f"GPT-2 Heads — concentration diagnostics (validation N={validation_n}; entropy N={n_docs})")
fig.tight_layout()
save_figure("concentration_diagnostics")
plt.show()
plt.close(fig)


## Boundary visibility and query position
The boundary is causally invisible to paragraph-1 queries. Compare other-query,
post-boundary and ≥32-positions-past-boundary mass before interpreting preference.
Entropy position bins report their actual number of contributing documents.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for metric, color, label in [
        ("boundary_from_other", BLUE, "all non-boundary queries"),
        ("boundary_from_exposed", PURPLE, "post-boundary queries"),
        ("boundary_from_far", GREEN, "≥32 positions after boundary")]:
    head_scatter(axes[0], np.nanmean(main[metric], axis=0), color, label)
axes[0].set_ylabel("Attention to natural paragraph boundary")
axes[0].legend(fontsize=7)
axes[0].set_title("Boundary visibility")
position_means = np.nanmean(main["position_entropy"], axis=0).mean(axis=1)
bin_documents = (main["position_counts"] > 0).sum(0)
labels = ["1–16", "17–32", "33–64", "65–128", "129–256", "257–512", "513–1023"]
sns.heatmap(position_means, ax=axes[1], vmin=0, vmax=1, cmap="viridis",
            xticklabels=[f"{b}\n(n={n})" for b, n in zip(labels, bin_documents)],
            yticklabels=range(1, 13), cbar_kws={"label": "Normalized entropy"})
axes[1].tick_params(axis="x", rotation=45)
axes[1].tick_params(axis="y", rotation=0)
axes[1].set_ylabel("Layer")
axes[1].set_xlabel("Query-position bin (document counts)")
fig.suptitle(f"GPT-2 Heads — causal-support diagnostics (N={n_docs})")
fig.tight_layout()
save_figure("boundary_and_position")
plt.show()
plt.close(fig)


## Figure 1: actual examples from the measured GPT-2 heads
The fixed illustration is candidate 0, not a best-scoring validation example.
Each column uses the same tokens: position 0 plus a window around the paragraph
boundary. Lines preserve the original attention values; omitted keys are not
renormalized. The displayed-key mass is reported per panel. A second figure
shows full-sequence matrices so the cropped link diagram can be checked.


In [ ]:
example = dict(np.load(RESULTS_DIR / "canonical_example_heads.npz"))
boundary_start = int(example["boundary_positions"][0])
window_start = max(1, boundary_start - 10)
window_stop = min(len(example["tokens"]), boundary_start + 14)
positions = [0] + list(range(window_start, window_stop))
titles = ["L03H07: periods/newlines", "L11H04: boundary candidate",
          "L08H03: initial-key sink", "L05H12: previous-token attention",
          "L01H12: broad attention"]
fig, axes = plt.subplots(1, 5, figsize=(19, 7))
for panel, ax in enumerate(axes):
    weights = example["attention"][panel].astype(float)
    labels = [str(example["tokens"][q]).replace("<|endoftext|>", "<EOT>")
              for q in positions]
    for row, (q, label) in enumerate(zip(positions, labels)):
        color = RED if q in example["boundary_positions"] or q == 0 else BLACK
        ax.text(-0.05, -row, f"{q}:{label}", ha="right", va="center",
                fontsize=7, color=color)
        ax.text(1.05, -row, label, ha="left", va="center", fontsize=7, color=color)
    for row, q in enumerate(positions):
        for col, k in enumerate(positions):
            value = float(weights[q, k])
            if value >= 0.003:
                ax.plot([0, 1], [-row, -col], color=BLUE,
                        alpha=min(value, 1), linewidth=1)
    visible_mass = weights[np.ix_(positions[1:], positions)].sum(-1).mean()
    ax.set_title(titles[panel] + f"\nshown-key mass={visible_mass:.2f}", fontsize=9)
    ax.set_xlim(-0.55, 1.45)
    ax.set_ylim(-len(positions), 1)
    ax.axis("off")
fig.suptitle(f"GPT-2 Heads — Figure 1 analogue; sample 0 from N={n_docs}", y=1.01)
fig.tight_layout()
save_figure("figure1_head_examples")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 5, figsize=(17, 3.7))
for panel, ax in enumerate(axes):
    im = ax.imshow(example["attention"][panel], cmap="magma", vmin=0, vmax=1,
                   aspect="equal", interpolation="nearest")
    ax.axvline(boundary_start, color="cyan", linewidth=0.6, linestyle="--")
    ax.set_title(titles[panel], fontsize=8)
    ax.set_xlabel("Key position")
    if panel == 0:
        ax.set_ylabel("Query position")
fig.colorbar(im, ax=axes.tolist(), shrink=0.7, label="Original attention weight")
fig.suptitle(f"GPT-2 Heads — full sample-0 maps (N={n_docs})", y=1.08)
save_figure("full_head_examples")
plt.show()
plt.close(fig)


## Section 6 (optional; independent of Experiment 2/3)
Raw `entropies` remains available above. Clustering requires distances computed
over the matching input corpus; it is not automatically run here. No BERT
linguistic head annotations are transferred to GPT-2.


In [ ]:
RUN_CLUSTERING = False
if RUN_CLUSTERING:
    import pickle
    from sklearn.manifold import MDS
    with open("data/head_distances.pkl", "rb") as f:
        js_divergences = pickle.load(f)
    embedding = MDS(dissimilarity="precomputed", random_state=42).fit_transform(js_divergences)
    plt.figure(figsize=(6, 5))
    plt.scatter(embedding[:, 0], embedding[:, 1],
                c=np.repeat(np.arange(1, 13), 12), cmap="viridis")
    plt.colorbar(label="Layer")
    plt.title(f"GPT-2 Heads: supplied Section-6 distances (verify corpus; N={n_docs})")
    save_figure("section6_optional")
    plt.show()
